# 구글 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!pip install -qU langchain-text-splitters langchain-community langgraph langchain-huggingface faiss-cpu bitsandbytes accelerate PyPDF2

In [27]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# pdf parser
- 최대 100장씩 바꿀 수 있어서 5개의 pdf로 나눔

In [8]:
from pypdf import PdfReader, PdfWriter
import os
import glob

# 작업 디렉토리 설정
work_dir = '/Users/yoonwogud/Desktop/Git/DeepLearning/Part3'
os.chdir(work_dir)

# 원본 파일 찾기 (파일명 인코딩 이슈 방지를 위해 glob 사용)
# 이미 분할된 파일(_part)은 제외
files = [f for f in glob.glob("2024년+원천징수의무자를+위한+연말정산+신고안내.pdf") if "_part" not in f]

if not files:
    print("대상 PDF 파일을 찾을 수 없습니다.")
else:
    source_path = files[0]
    print(f"처리할 파일: {source_path}")
    
    base_name = os.path.splitext(source_path)[0]

    try:
        reader = PdfReader(source_path)
        total_pages = len(reader.pages)
        chunk_size = 100

        print(f"총 페이지 수: {total_pages}")

        for i in range(0, total_pages, chunk_size):
            writer = PdfWriter()
            start_page = i
            end_page = min(i + chunk_size, total_pages)
            
            for page_num in range(start_page, end_page):
                writer.add_page(reader.pages[page_num])
            
            # 파트 번호 (1부터 시작)
            part_num = i // chunk_size + 1
            output_filename = f"{base_name}_part{part_num}.pdf"
            
            with open(output_filename, "wb") as f:
                writer.write(f)
            print(f"생성 완료: {output_filename} (페이지 {start_page+1}~{end_page})")

    except Exception as e:
        print(f"에러 발생: {e}")


처리할 파일: 2024년+원천징수의무자를+위한+연말정산+신고안내.pdf
총 페이지 수: 426
생성 완료: 2024년+원천징수의무자를+위한+연말정산+신고안내_part1.pdf (페이지 1~100)
생성 완료: 2024년+원천징수의무자를+위한+연말정산+신고안내_part2.pdf (페이지 101~200)
생성 완료: 2024년+원천징수의무자를+위한+연말정산+신고안내_part3.pdf (페이지 201~300)
생성 완료: 2024년+원천징수의무자를+위한+연말정산+신고안내_part4.pdf (페이지 301~400)
생성 완료: 2024년+원천징수의무자를+위한+연말정산+신고안내_part5.pdf (페이지 401~426)


In [ ]:
import requests, glob, json, os
from getpass import getpass

upstage_api_key = getpass("upstage API Key를 입력해주세요: ")
url = "https://api.upstage.ai/v1/document-digitization"
headers = {"Authorization": f"Bearer {upstage_api_key}"}

target_files = glob.glob("*part5.pdf")

if not target_files:
    print("파일을 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    file_path = target_files[0]
    print(f"분석 시작: {file_path}")

    # 2. API 요청
    with open(file_path, "rb") as f:
        files = {"document": f}
        data = {
            "model": "document-parse-250618",
            "ocr": "auto",
            "chart_recognition": True,
            "coordinates": True,
            "output_formats": '["markdown"]',
            "base64_encoding": '["figure"]',
        }
        
        # 100페이지라 시간이 좀 걸리므로 timeout 설정
        response = requests.post(url, headers=headers, files=files, data=data, timeout=300)

    # 3. 결과 저장
    if response.status_code == 200:
        result = response.json()
        
        # 결과를 JSON 파일로 저장 (나중에 불러와서 쓰기 편함)
        output_filename = "result_part5.json"
        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
            
        print(f"완료! 결과가 '{output_filename}'에 저장되었습니다.")
    else:
        print(f"에러 발생: {response.status_code}")
        print(response.text)

분석 시작: 2024년+원천징수의무자를+위한+연말정산+신고안내_part5.pdf
완료! 결과가 'result_part5.json'에 저장되었습니다.


# 문서 청킹

In [ ]:
import json
import re
from transformers import AutoTokenizer
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter


def preprocess_markdown(text):
    text = re.sub(r'!\[image\]\(.*?\)', '', text)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    text = text.replace("국세청\nwww.nts.go.kr", "")
    text = text.replace("www.nts.go.kr", "")
    text = re.sub(r'\n{3,}', '\n\n', text)

    split_keyword = "# 머 리 말"
    if split_keyword in text:
        text = text.split(split_keyword)[1] # 뒷부분만 씀
        text = split_keyword + text # "머 리 말" 제목은 다시 붙여줌
        
    return text
    return text

language_model_name = "nlpai-lab/KULLM3"
tokenizer = AutoTokenizer.from_pretrained(language_model_name)

def token_counter(text):
    return len(tokenizer.encode(text))

with open("result_part1.json", "r", encoding="utf-8") as f:
    data = json.load(f)
markdown_text = data['content']['markdown']

clean_markdown = preprocess_markdown(markdown_text)

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(clean_markdown)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=token_counter
)

final_splits = text_splitter.split_documents(md_header_splits)

print(f"총 청크 개수: {len(final_splits)}")
if len(final_splits) > 2:
    print(final_splits[2].page_content)
    print(final_splits[2].metadata)


총 청크 개수: 347
연말정산 서비스  
국세청  
| 이용자 | 서비스 | 내 용 | 접근 경로 |
| --- | --- | --- | --- |
| 공통 | 원천징수(연말정산)안내 홈페이지 | 원천징수(연말정산)안내 홈페이지 | 국세청 홈페이지() 국세신고안내 → 개인 또는 법인 → 연말정산 |
| 공통 | 연말정산 관련 질의회신 및 판례 조회 | 연말정산 관련 질의회신 및 판례 조회 | 국세법령정보시스템 (www.hometax.go.kr → 법령정보) |
| 공통 | 인터넷 상담 및 회신 | 인터넷 상담 및 회신 | 국세청 국세상담센터 (www.hometax.go.kr → 상담/제보) |
| 근로자 | 소득공제 자료조회 | 연말정산 소득·세액공제 자료 조회 | 홈택스 → 장려금·연말정산·기부금 → 연말정산간소화 [안내] 126-내선1-5번 |
{'Header 1': '국세청'}


# Vector DB에 저장

In [ ]:
import faiss
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS


# embedding_model_name = "nlpai-lab/KURE-v1"
embedding_model_name = "nlpai-lab/KoE5"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
)

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
ids = vector_store.add_documents(documents=all_splits)

retriever = vector_store.as_retriever()

# 모델 및 토크나이저 세팅

In [ ]:


language_model_name = "nlpai-lab/KULLM3"
# language_model_name = "Bllossom/llama-3.2-Korean-Bllossom-AICA-5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True,
)
model = AutoModelForCausalLM.from_pretrained(
    language_model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
)
language_model_name = "nlpai-lab/KULLM3"
tokenizer = AutoTokenizer.from_pretrained(language_model_name)

In [ ]:
from transformers import pipeline

llm_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    do_sample=True,
    temperature=0.2,
    repetition_penalty=1.2,
    return_full_text=False,
    max_new_tokens=1000,
)
llm = HuggingFacePipeline(pipeline=llm_pipeline)
chat_model = ChatHuggingFace(llm=llm)

# RAG

In [ ]:
from langchain_core.prompts import PromptTemplate


template = """
아래에 주어진 맥락을 이용해 질문에 대해 답변해 줘.
주어진 맥락으로 답변이 어려운 상황이라면, 그냥 모른다고 답하면 되고 억지로 답변을 꾸며 내지 마.
최대한 자세하게 답변해 줘.
반드시 한국어로 답변해야 해.

맥락:
{context}

질문:
{question}
"""

prompt = PromptTemplate.from_template(template)

In [ ]:
# from langchain.schema.runnable import RunnablePassthrough
# from langchain.schema.output_parser import StrOutputParser

def format_docs(docs):
    print(docs)
    return "\n\n".join(doc.page_content for doc in docs)

retrieval_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
question = "연말 정산 때 비거주자가 주의할 점을 알려 줘."

result = retrieval_chain.invoke(question)
print(result)

In [ ]:
question = "2024년 개정 세법 중에 월세와 관련한 내용이 있을까?"

result = retrieval_chain.invoke(question)
print(result)

In [ ]:
question = "기부금 공제 때 주의할 점은?"

result = retrieval_chain.invoke(question)
print(result)